In [1]:
csv_files=[
    ('customers.csv', 'customers'),
    ('orders.csv', 'orders'),
    ('sellers.csv', 'sellers'),
    ('products.csv', 'products'),
    ('order_items.csv', 'order_items'),
    ('payments.csv', 'payments'),
    ('geolocation.csv','geolocation')  # Added payments.csv for specific handling
]

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import mysql.connector

In [3]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")
sellers = pd.read_csv("sellers.csv")
geolocation=pd.read_csv('geolocation.csv')
payments=pd.read_csv('payments.csv')
order_items=pd.read_csv('order_items.csv')


In [4]:
import os

In [6]:
print(products.head())

                         product_id product category  product_name_length  \
0  1e9e8ef04dbcff4541ed26657ea517e5        perfumery                 40.0   
1  3aa071139cb16b67ca9e5dea641aaa2f              Art                 44.0   
2  96bd76ec8810374ed1b65e291975717f    sport leisure                 46.0   
3  cef67bcfe19066a932b7673e239eb23d           babies                 27.0   
4  9dc1a7de274444849c219cff195d0b71       housewares                 37.0   

   product_description_length  product_photos_qty  product_weight_g  \
0                       287.0                 1.0             225.0   
1                       276.0                 1.0            1000.0   
2                       250.0                 1.0             154.0   
3                       261.0                 1.0             371.0   
4                       402.0                 4.0             625.0   

   product_length_cm  product_height_cm  product_width_cm  
0               16.0               10.0           

In [7]:
df = customers.drop_duplicates()

In [8]:
customers.dropna()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


In [9]:
pip install pyodbc

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [2]:
import pyodbc

conn = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=localhost\\SQLEXPRESS;'
    'DATABASE=database1;'
    'Trusted_Connection=yes;'
)

print("Connected successfully!")
cursor=conn.cursor()

Connected successfully!


In [36]:
print(type(cursor))

<class 'pyodbc.Cursor'>


In [39]:
df = df.where(pd.notnull(df), None)

In [47]:
import pandas as pd
import os

def get_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return 'INT'
    elif pd.api.types.is_float_dtype(dtype):
        return 'FLOAT'
    elif pd.api.types.is_bool_dtype(dtype):
        return 'BIT'
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return 'DATETIME'
    else:
        return 'VARCHAR(MAX)'


folder_path = "c:/Users/nandi/Desktop/PythonSQL"

for csv_file, table_name in csv_files:

    file_path = os.path.join(folder_path, csv_file)

    df = pd.read_csv(file_path)

    # Replace NaN with None
    df = df.where(pd.notnull(df), None)

    print(f"Processing {csv_file}")
    print(f"NaN values:\n{df.isnull().sum()}\n")

    # Clean column names
    df.columns = [
        col.replace(' ', '_').replace('-', '_').replace('.', '_')
        for col in df.columns
    ]

    # -------------------------
    # CREATE TABLE
    # -------------------------
    columns = ', '.join(
        [f'[{col}] {get_sql_type(df[col].dtype)}' for col in df.columns]
    )

    create_table_query = f"""
    IF NOT EXISTS (
        SELECT * FROM sys.tables WHERE name = '{table_name}'
    )
    CREATE TABLE {table_name} (
        {columns}
    )
    """

    cursor.execute(create_table_query)
    conn.commit()

    # -------------------------
    # INSERT DATA
    # -------------------------
    insert_sql = f"""
    INSERT INTO {table_name}
    ({', '.join(['[' + col + ']' for col in df.columns])})
    VALUES ({', '.join(['?'] * len(df.columns))})
    """

    for _, row in df.iterrows():

        values = tuple(None if pd.isna(x) else x for x in row)

        cursor.execute(insert_sql, values)

    conn.commit()

conn.close()

Processing customers.csv
NaN values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Processing orders.csv
NaN values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Processing sellers.csv
NaN values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

Processing products.csv
NaN values:
product_id                      0
product category              610
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_heig

KeyboardInterrupt: 

In [49]:
df['product_id'].str.len().max()

KeyError: 'product_id'

In [50]:
df = pd.read_csv("products.csv")

print(df.columns)
print(df.iloc[:,5].unique())

Index(['product_id', 'product category', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='object')
[ 225. 1000.  154. ... 1206.  934. 1920.]


In [51]:
df['product_id'].isnull().sum()

np.int64(0)

 List all unique cities where customers are located

In [52]:
query=""" select distinct customer_city from customers"""
cursor.execute(query)
data=cursor.fetchall()
data

[('sao joao de meriti',),
 ('rio das ostras',),
 ('pindamonhangaba',),
 ('atilio vivacqua',),
 ('bady bassitt',),
 ('iturama',),
 ('bom jesus da lapa',),
 ('surubim',),
 ('angical',),
 ('santa rosa de viterbo',),
 ('castelo',),
 ('jacinto machado',),
 ('caraiba',),
 ('esperantinopolis',),
 ('jundiai do sul',),
 ('pradopolis',),
 ('juripiranga',),
 ('capela de santana',),
 ('itaguari',),
 ('mesquita',),
 ('cachoeira do brumado',),
 ('araguana',),
 ('guarda-mor',),
 ('santa maria da vitoria',),
 ('miguel calmon',),
 ('santa filomena',),
 ('paulistana',),
 ('bodoquena',),
 ('aparecida de goiania',),
 ('sao joao do piaui',),
 ('cicero dantas',),
 ('lambari',),
 ('utinga',),
 ('carmo da cachoeira',),
 ('bela cruz',),
 ('mambai',),
 ('ataleia',),
 ('ouvidor',),
 ('pau brasil',),
 ('gravata',),
 ('angra dos reis',),
 ('santa maria madalena',),
 ('conceicao do para',),
 ('parnarama',),
 ('jacutinga',),
 ('conceicao do tocantins',),
 ('escada',),
 ('mato verde',),
 ('papagaios',),
 ('rio claro'

count number of orders places in 2017


In [53]:
query=""" select count(order_id) from orders 
where year(order_purchase_timestamp) = 2017"""
cursor.execute(query)
data=cursor.fetchall()
data[0][0]

496111

<h1> Find the total sales per category 

In [3]:
query=""" select products.product_category ,sum(payments.payment_value) 
from products join order_items on products.product_id= order_items.product_id
join payments on payments.order_id=order_items.order_id
group by product_category"""
cursor.execute(query)
data=cursor.fetchall()
data

[('fixed telephony', 8694430.92000001),
 ('House comfort', 3541820.5200000713),
 ('Cool Stuff', 32747316.000000086),
 ('ELECTRICES 2', 5231665.319999998),
 ('Construction Tools Construction', 10141976.460000006),
 ("Fashion Children's Clothing", 32998.139999999985),
 ('Casa Construcao', 5739102.180000001),
 ('Fashion Underwear and Beach Fashion', 534010.6799999998),
 ('Fashion Calcados', 1361755.0799999998),
 ('Construction Tools Illumination', 3053924.1599999936),
 ('climatization', 3829167.7199999965),
 ('automotive', 35796361.86000008),
 ('pet Shop', 13073296.740000002),
 ('PCs', 11723105.099999983),
 ('Games consoles', 8210175.960000008),
 ('Industry Commerce and Business', 2383321.9199999953),
 ('Market Place', 1904108.6400000006),
 ('Arts and Crafts', 97699.1400000001),
 ('bed table bath', 71927254.13999993),
 ('Construction Tools Tools', 884900.9399999996),
 ('musical instruments', 9789113.040000016),
 ('drinks', 3000045.2999999993),
 ('flowers', 92946.41999999987),
 ('Hygiene d

<h1>Calculate the percentages of orders that were paid in installments


In [ ]:
query=""" """
cursor.execute(query)
data=cursor.fetchall()
data